In [ ]:
import draw_arbors_AM as draw

import math
import networkx as nx
import numpy as np
import plotly.graph_objs as go
import pylab
import plant_gravitropism as pg

last_day_files = pg.get_last_day_files()
print(len(last_day_files))

In [ ]:
fname = last_day_files[1]
alphas = np.linspace(0, 1, 15)
evaluated = []

for a in alphas:
    draw.plot_arbors(fname, 0, a)
    evaluated.append(pg.evaluate_parameters(fname, 0, a))


In [ ]:
orthogonal_dists = [float(items[2]) for items in evaluated]
print(orthogonal_dists)

sq_orthogonal_dists = [float(items[3]) for items in evaluated]
print(sq_orthogonal_dists)



In [ ]:
# 
optimal_orth_dist = min(orthogonal_dists)
optimal_orth_dist_index = orthogonal_dists.index(optimal_orth_dist)

optimal_orth_alpha = alphas[optimal_orth_dist_index]
print(f"Minimum Orthogonal Distance → {optimal_orth_dist}")
print(f"Optimal alpha value (Orthogonal Distance) → {optimal_orth_alpha}\n")

optimal_sq_orth_dist = min(sq_orthogonal_dists)
optimal_sq_orth_dist_index = sq_orthogonal_dists.index(optimal_sq_orth_dist)

optimal_sq_alpha = alphas[optimal_sq_orth_dist_index]
print(f"Minimum Squared Orthogonal Distance → {optimal_sq_orth_dist}")
print(f"Optimal alpha value (Squared Orthogonal Distance) → {optimal_sq_alpha}")


In [ ]:
# first draft of pipeline function
def optimal_alpha_sq_orthogonal(filename, steps):
    alphas = np.linspace(0, 1, steps)
    evaluate_values = []

    for a in alphas:
        evaluate_values.append(pg.evaluate_parameters(filename, 0, a))        #  DEPENDENCY --> plant_gravitropism
    
    wiring_costs = [float(item[0]) for item in evaluate_values]
    delays = [float(item[1]) for item in evaluate_values]
    sq_orthogonal_dists = [float(item[3]) for item in evaluate_values]

    optimal_sq_orth_dist = min(sq_orthogonal_dists)

    optimal_sq_orth_dist_index = sq_orthogonal_dists.index(optimal_sq_orth_dist)
    optimal_sq_alpha = alphas[optimal_sq_orth_dist_index]

    wiring_cost = wiring_costs[optimal_sq_orth_dist_index]
    delay = delays[optimal_sq_orth_dist_index]

    return optimal_sq_orth_dist, optimal_sq_alpha, wiring_cost, delay



def optimal_alpha_orthogonal(filename, steps):
    """
    Determines the optimal alpha value based on the orthogonal distance rather than the squared orthogonal distance
    """
    alphas = np.linspace(0, 1, steps)
    evaluate_values = []

    for a in alphas:
        evaluate_values.append(pg.evaluate_parameters(filename, 0, a))        #  DEPENDENCY --> plant_gravitropism
    
    wiring_costs = [float(item[0]) for item in evaluate_values]
    delays = [float(item[1]) for item in evaluate_values]
    orthogonal_dists = [float(item[2]) for item in evaluate_values]

    optimal_orth_dist = min(orthogonal_dists)

    optimal_orth_dist_index = orthogonal_dists.index(optimal_orth_dist)
    optimal_orth_alpha = alphas[optimal_orth_dist_index]

    wiring_cost = wiring_costs[optimal_orth_dist_index]
    delay = delays[optimal_orth_dist_index]

    return optimal_orth_dist, optimal_orth_alpha, wiring_cost, delay

In [ ]:
opt_distance, opt_alpha, wiring_cost, delay = optimal_alpha_sq_orthogonal(last_day_files[1], 100)
print(f"Values for {last_day_files[1]}: Optimal Alpha → {opt_alpha} \nOptimal Distance Value → {opt_distance} \nWiring Cost → {wiring_cost} \nDelay → {delay}")


In [ ]:
# Double-checking evaluated parameter values with generated csv values to ensure code accuracy

alphas = np.round(np.arange(0, 1.01, 0.01), 2)
filename = last_day_files[0]


evaluate_values = []

for a in alphas:
    evaluate_values.append(pg.evaluate_parameters(filename, 0, a))        #  DEPENDENCY --> plant_gravitropism

wiring_costs = [float(item[0]) for item in evaluate_values]
delays = [float(item[1]) for item in evaluate_values]
orthogonal_dists = [float(item[2]) for item in evaluate_values]
sq_orthogonal_dists = [float(item[3]) for item in evaluate_values]


print(filename)

print(orthogonal_dists)

print(sq_orthogonal_dists)

print(alphas)

print(wiring_costs)

print(delays)

In [ ]:
# To create the finalized version of the function for this pipeline

# parameters: arbors to find optimal distance of (will be all csv files in last_day_files), step size for alpha value
# output: a csv containing name of the arbor, optimal alpha, wiring cost of α, delay of α, orthogonal distance of α, and squared orthogonal distance of α


In [ ]:
name = last_day_files[0] + "hi"
print(name)

In [ ]:
array = [0, 1, 2, 34, 5, 6, 67 ,8]
print(array[0:5])
print(array[5:])

Testing Claude-generated code on toy networks

In [1]:
import math
from math import sqrt, log
from utils import *
import networkx as nx
from scipy.spatial.distance import euclidean
import numpy as np
from read_arbor_reconstruction import read_arbor_full, read_arbor_full_initial
from constants import *
from optimal_midpoint import optimal_midpoint, optimal_midpoint_approx, optimal_midpoint_alpha1
from collections import defaultdict, namedtuple
import seaborn as sns
import os
import pandas as pd
from scipy.optimize import minimize_scalar, fsolve
import pareto_functions as pf

import plotly.graph_objs as go # for plotting arbors
import plant_gravitropism as pg



CostSpec = namedtuple('CostSpec', ['wiring_transform', 'delay_transform'])
"""
VVVVVVVV originals VVVVVVVVV

def _homogeneous_wiring(curve, to_root): return curve
def _homogeneous_delay(curve, to_root): return curve + to_root

def _heterogeneous_wiring(curve, to_root): return curve ** 2
def _heterogeneous_delay(curve, to_root): return log(1 + curve) + log(1 + to_root)

"""
def _homogeneous_wiring(curve, to_root): return curve
def _homogeneous_delay(curve, to_root): 
    if (curve + to_root) < 0: 
        print("Delay is negative for HOMOGENEOUS") 
    return curve + to_root

def _heterogeneous_wiring(curve, to_root): return curve ** 2
def _heterogeneous_delay(curve, to_root): 
    if (log(1 + curve) + log(1 + to_root)) < 0: 
        print("Delay is negative for HOMOGENEOUS") 
    return log(1 + curve) + log(1 + to_root)

HOMOGENEOUS = CostSpec(
    wiring_transform = _homogeneous_wiring,
    delay_transform = _homogeneous_delay,
)

HETEROGENEOUS = CostSpec(
    wiring_transform = _heterogeneous_wiring, 
    delay_transform = _heterogeneous_delay,
)

COST_SPECS = {
    'homogeneous': HOMOGENEOUS,
    'heterogeneous': HETEROGENEOUS,
}





# Calculates length of the lateral root
def lateral_root_path_length(G, tip):
    """Sum edge lengths from tip back to main root insertion point."""
    length = 0
    visited = set()
    queue = [tip]
    while queue:
        node = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)
        for neighbor in G.neighbors(node):
            if neighbor not in visited:
                label = G.nodes[neighbor]['label']
                if label in ('lateral root', 'lateral root tip'):
                    length += G[node][neighbor]['length']
                    queue.append(neighbor)
                elif label in ('main root', 'main root base'):
                    length += G[node][neighbor]['length']
                    # stop here — this is the insertion point
    return length

# note: lateral_root_path_length requires a tip, which seems to be an object that can be 
# converted into a queue that is later traversed (elements are popped).

# Claude-generated conduction_delay function
def conduction_delay(G, cost_spec=HOMOGENEOUS):
    print("CONDUCTION DELAY")
    droot = {}
    queue = []
    visited = set()
    root = G.graph.get('main root base', G.graph.get('main root'))
    queue.append(root)
    droot[root] = 0
    delay = 0

    while len(queue) > 0:
        curr = queue.pop(0)
        assert curr not in visited
        visited.add(curr)

        if G.nodes[curr]['label'] == 'lateral root tip':
            curve = lateral_root_path_length(G, curr)
            to_root = droot[curr] - curve   # subtract lateral length to get main root distance
            delay += cost_spec.delay_transform(curve, to_root)

        for u in G.neighbors(curr):
            if u not in visited:
                queue.append(u)
                droot[u] = droot[curr] + G[curr][u]['length']

    assert len(visited) == G.number_of_nodes()
    return delay







# from plant_gravitropism

# ---- evaluate_parameters ----

# -- arbor_best_cost
def compute_main_root_base_distances(arbor):
    """
    Returns a dict mapping each main root node to its distance from the main root base,
    using edge 'length' attributes and the ordering from get_main_root_segments.
    """
    base = arbor.graph['main root base']

    base_dist = {base: 0}
    for seg_start, seg_end in get_main_root_segments(arbor):
        base_dist[seg_end] = base_dist[seg_start] + arbor[seg_start][seg_end]['length']

    return base_dist

def get_insertion_segment(arbor, lateral_tip, segments):
    """
    For a given lateral root tip, walk up the graph until hitting a main root node,
    then find which segment that insertion point belongs to.
    Returns all segments up to and including the insertion segment.

    Parameters
    ----------
    arbor : networkx.Graph
        Observed arbor graph.
    lateral_tip : tuple
        (x, y) of the lateral root tip.
    segments : list
        Ordered list of main root segments from get_main_root_segments.

    Returns
    -------
    list of segments up to and including the insertion segment
    """
    # BFS up from tip until we hit a main root node
    visited = set()
    queue = [lateral_tip]
    insertion_point = None

    while queue:
        node = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)
        label = arbor.nodes[node]['label']
        if label in ('main root', 'main root base'):
            insertion_point = node
            break
        for neighbor in arbor.neighbors(node):
            if neighbor not in visited:
                queue.append(neighbor)

    assert insertion_point is not None, f"No main root node found from tip {lateral_tip}"

    # Find the segment that contains the insertion point and return all up to it
    valid_segments = []
    for seg in segments:
        valid_segments.append(seg)
        if insertion_point in seg:
            break

    return valid_segments

# - optimize_tip - 
def is_between(a, x, b):
    """Returns True if x is strictly between a and b (in either order)."""
    return a < x < b or b < x < a

# find_best_cost_brute_force
def branch_point_from_t(x0, y0, x1, y1, t):
    """Linearly interpolate a branch point along segment (x0,y0)-(x1,y1) at parameter t."""
    return x0 + t * (x1 - x0), y0 + t * (y1 - y0)

# ... compute_cost ...
def curve_length(G, x0, y0, p, q):
    """
    Arc length of the parabola G*x^2 + b*x + c between (x0, y0) and (p, q).
    Uses closed-form solution when G != 0, Euclidean distance when G == 0.
    """
    if G == 0:
        # Straight line distance
        return euclidean((x0, y0), (p, q))

    # Shift to local frame: branch point becomes origin
    p_local = p - x0
    q_local = q - y0

    k = (q_local - G * p_local**2) / p_local

    theta_0 = math.atan(k)
    theta_p = math.atan(2 * G * p_local + k)

    def sec(theta):
        return 1.0 / math.cos(theta)

    def F(theta):
        return sec(theta) * math.tan(theta) + math.log(abs(sec(theta) + math.tan(theta)))

    return abs((1.0 / (4 * G)) * (F(theta_p) - F(theta_0)))


def compute_cost(alpha, G, seg_base_dist, t, seg_length, branch_x, branch_y, tip_x, tip_y, cost_spec=HOMOGENEOUS):
    """
    Compute total cost, wiring, and delay for a given branch point.

    Parameters
    ----------
    seg_base_dist : float
        Distance from root base to start of segment (x0, y0).
    t : float
        Position along segment [0, 1].
    seg_length : float
        Length of the main root segment.
    branch_x, branch_y : float
        Coordinates of the branch point on the main root.
    tip_x, tip_y : float
        Coordinates of the lateral root tip.
    """
    curve = curve_length(G, branch_x, branch_y, tip_x, tip_y)
    to_root = seg_base_dist + t * seg_length
    wiring = cost_spec.wiring_transform(curve, to_root)
    delay = cost_spec.delay_transform(curve, to_root)
    cost = alpha * wiring + (1 - alpha) * delay
    return cost, wiring, delay


def find_best_cost_brute_force(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=HOMOGENEOUS):
    """
    Find the optimal branch point on segment (x0,y0)-(x1,y1) for lateral tip (p, q)
    by brute-force search over t in [0, 1] with step 0.01.

    Returns
    -------
    tuple : (cost, wiring, delay, best_t, best_x, best_y, p, q)
    """
    seg_length = euclidean((x0, y0), (x1, y1))
    best_cost = math.inf
    best_wiring = math.inf
    best_delay = math.inf
    best_t = None
    best_x = None
    best_y = None

    for t in pylab.arange(0, 1 + 0.01, 0.01):
        branch_x, branch_y = branch_point_from_t(x0, y0, x1, y1, t)
        cost, wiring, delay = compute_cost(
            alpha, G, seg_base_dist, t, seg_length, branch_x, branch_y, p, q, cost_spec=cost_spec 
        )
        if cost <= best_cost:
            best_cost = cost
            best_wiring = wiring
            best_delay = delay
            best_t = t
            best_x = branch_x
            best_y = branch_y

    return best_cost, best_wiring, best_delay, best_t, best_x, best_y, p, q


def find_best_cost_brent(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=HOMOGENEOUS):
    """
    Find the optimal branch point on segment (x0,y0)-(x1,y1) for lateral tip (p, q)
    using Brent's method to directly minimize the cost function.

    For G=0, falls back to exact analytical solution from optimal_midpoint.py.
    For G!=0, uses minimize_scalar with method='bounded' on [0, 1].

    Returns
    -------
    tuple : (cost, wiring, delay, best_t, best_x, best_y, p, q)
    """
    seg_length = euclidean((x0, y0), (x1, y1))

    # G != 0: minimize cost directly using Brent's method
    def cost_at_t(t):
        branch_x, branch_y = branch_point_from_t(x0, y0, x1, y1, t)
        c, _, _ = compute_cost(alpha, G, seg_base_dist, t, seg_length, branch_x, branch_y, p, q, cost_spec=cost_spec)
        return c

    result = minimize_scalar(cost_at_t, bounds=(0, 1), method='bounded')
    best_t = result.x
    best_x, best_y = branch_point_from_t(x0, y0, x1, y1, best_t)
    best_cost, best_wiring, best_delay = compute_cost(
        alpha, G, seg_base_dist, best_t, seg_length, best_x, best_y, p, q, cost_spec=cost_spec
    )

    return best_cost, best_wiring, best_delay, best_t, best_x, best_y, p, q


# find_best_cost_analytical
def make_costprime(G, alpha, l, theta, p, q):
    """
    Returns the derivative of cost w.r.t. t for the analytical case.

    Parameters
    ----------
    G : float
        Gravity parameter.
    alpha : float
        Weighting parameter.
    l : float
        Length of the main root segment.
    theta : float
        Angle of the segment.
    p, q : float
        Coordinates of the lateral root tip (shifted to local frame).
    """
    A1 = G * (l * math.cos(theta))**2
    B1 = -l * math.sin(theta)
    C1 = q - G * p**2
    D1 = -l * math.cos(theta)
    E1 = p

    def b(t):
        return (A1*t**2 + B1*t + C1) / (D1*t + E1)

    def bprime(t):
        num = (D1*t + E1) * (2*A1*t + B1) - D1 * (A1*t**2 + B1*t + C1)
        den = (D1*t + E1)**2
        return num / den

    def costprime(t):
        bt = b(t)
        term1 = (bprime(t) / (2 * G)) * (
            math.sqrt(1 + (2*G*p + bt)**2) -
            math.sqrt(1 + (2*G * t * l * math.cos(theta) + bt)**2)
        )
        term2 = (1 - alpha) * l
        term3 = math.sqrt(1 + (2*G * t * l * math.cos(theta) + bt)**2) * l * math.cos(theta)
        return term1 + term2 - term3

    return costprime

def find_root_in_unit_interval(func, num_guesses=1):
    """
    Find roots of func in [0, 1] using fsolve with evenly spaced initial guesses,
    excluding endpoints.
    """
    roots = []
    guesses = np.linspace(0, 1, num_guesses + 2)[1:-1]
    for guess in guesses:
        try:
            root = fsolve(func, guess)[0]
            if 0 <= root <= 1 and not any(np.isclose(root, r) for r in roots):
                roots.append(root)
        except Exception:
            pass
    return sorted(roots)

def find_best_cost_analytical(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=HOMOGENEOUS):
    """
    Find the optimal branch point on segment (x0,y0)-(x1,y1) for lateral tip (p, q)
    using the analytical derivative of the cost function.

    For G=0, uses exact analytical solution from optimal_midpoint.py.
    For alpha=1, minimizes curve length directly via scipy minimize_scalar.
    Otherwise, uses analytical costprime approach with fsolve.

    Returns
    -------
    tuple : (cost, wiring, delay, best_t, best_x, best_y, p, q)
    """

    seg_length = euclidean((x0, y0), (x1, y1))

    # G=0 case: use exact analytical solution from optimal_midpoint.py
    if G == 0:
        if alpha == 1:
            cost_val, (best_x, best_y), best_t = optimal_midpoint.optimal_midpoint_alpha1(
                (x0, y0), (x1, y1), (p, q)
            )
        else:
            cost_val, (best_x, best_y), best_t = optimal_midpoint.optimal_midpoint_exact(
                (x0, y0), (x1, y1), (p, q), alpha, seg_base_dist
            )
        wiring = euclidean((best_x, best_y), (p, q))
        to_root = seg_base_dist + best_t * seg_length
        delay = wiring + to_root
        return cost_val, wiring, delay, best_t, best_x, best_y, p, q
    # G != 0: use analytical costprime with fsolve
    else:
        # theta = math.atan2(abs(y1 - y0), abs(x1 - x0)) if (x1 != x0 and y1 != y0) else 0
        theta = math.atan2(y1 - y0, x1 - x0)
        p_local = p - x0
        q_local = q - y0

        costprime = make_costprime(G, alpha, seg_length, theta, p_local, q_local)

        def cost_at_t(t):
            branch_x, branch_y = branch_point_from_t(x0, y0, x1, y1, t)
            c, _, _ = compute_cost(alpha, G, seg_base_dist, t, seg_length, branch_x, branch_y, p, q, cost_spec=cost_spec)
            return c

        roots = find_root_in_unit_interval(costprime)
        valid_roots = [r for r in roots if 0 <= r <= 1]

        if valid_roots:
            best_t = min(valid_roots, key=cost_at_t)
        else:
            best_t = 0.0 if cost_at_t(0) <= cost_at_t(1) else 1.0

        best_x, best_y = branch_point_from_t(x0, y0, x1, y1, best_t)
        best_cost, best_wiring, best_delay = compute_cost(
            alpha, G, seg_base_dist, best_t, seg_length, best_x, best_y, p, q, cost_spec=cost_spec
        )

        return best_cost, best_wiring, best_delay, best_t, best_x, best_y, p, q


OPTIMIZATION_METHOD = 'brent'
def optimize_tip(tip, segments, base_dist, alpha, G, cost_spec=HOMOGENEOUS):
    p, q = tip
    results = []

    for seg in segments:
        x0, y0 = seg[0]
        x1, y1 = seg[1]
        seg_base_dist = base_dist[(x0, y0)]

        if is_between(x0, p, x1) or OPTIMIZATION_METHOD == 'brute_force':
            result = find_best_cost_brute_force(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=cost_spec)
        elif OPTIMIZATION_METHOD == 'brent':
            result = find_best_cost_brent(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=cost_spec)
        else:
            result = find_best_cost_analytical(alpha, G, seg_base_dist, x0, y0, x1, y1, p, q, cost_spec=cost_spec)

        results.append(result)

    best = min(results)
    return best

def arbor_best_cost(arbor, G, alpha, cost_spec=HOMOGENEOUS):
    """
    For each lateral root tip in the arbor, find the optimal branch point
    on the main root under the given (G, alpha) parameters.

    Parameters
    ----------
    arbor : networkx.Graph
        Already-loaded observed arbor graph.
    G : float
        Gravity parameter.
    alpha : float
        Weighting parameter.

    Returns
    -------
    list of tuples : [(cost, wiring, delay, best_t, best_x, best_y, tip_x, tip_y), ...]
    """
    segments = get_main_root_segments(arbor)
    base_dist = compute_main_root_base_distances(arbor)

    lat_tips = [
        node for node in arbor.nodes()
        if arbor.nodes[node]['label'] == 'lateral root tip'
    ]

    final = []
    for tip in lat_tips:
        valid_segments = get_insertion_segment(arbor, tip, segments)
        result = optimize_tip(tip, valid_segments, base_dist, alpha, G, cost_spec=cost_spec)
        if result is not None:
            final.append(result)
        else:
            print(f"Warning: No valid results for lateral tip at {tip}")

    return final


# calculate_orthogonal_errors

def collect_lateral_root_points(arbor, lateral_tip):
    """
    BFS from lateral_tip through 'lateral root' and 'lateral root tip' nodes.
    """
    lateral_points = []
    visited = set()
    queue = [lateral_tip]

    while queue:
        node = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)
        label = arbor.nodes[node]['label']
        if label in ('lateral root', 'lateral root tip'):
            lateral_points.append(node)
            for neighbor in arbor.neighbors(node):
                if neighbor not in visited and arbor.nodes[neighbor]['label'] in ('lateral root', 'lateral root tip'):
                    queue.append(neighbor)

    return lateral_points


def calc_coeff(G, x, y, p, q):
    """
    Returns coefficients b, c of the parabola G*x^2 + b*x + c
    that passes through (x, y) and (p, q).
    """
    b = (q - y - G * (p*p - x*x)) / (p - x)
    c = q - G * p*p - b * p
    return b, c

def collect_lateral_root_segments(arbor, lateral_tip):
    """
    Return list of (x0, y0, x1, y1) segments along the lateral root path
    from tip back to the main root insertion point.
    """
    segments = []
    path = collect_lateral_root_points(arbor, lateral_tip)  # existing BFS
    for i in range(len(path) - 1):
        x0, y0 = path[i]
        x1, y1 = path[i + 1]
        segments.append((x0, y0, x1, y1))
    return segments

def calculate_orthogonal_errors(gravity, arbor, main_root_pt, lateral_tip,
                                 n_subsample=100):
    """
    Compute total orthogonal distance and total squared orthogonal distance
    between the fitted parabola and the lateral root path.

    Sub-discretizes each lateral root segment into n_subsample points
    so that lightly-traced lateral roots are treated consistently with
    densely-traced ones.
    """
    px, py = main_root_pt
    tip_x, tip_y = lateral_tip

    b, c = calc_coeff(gravity, px, py, tip_x, tip_y)
    x_start, x_end = min(px, tip_x), max(px, tip_x)

    segments = collect_lateral_root_segments(arbor, lateral_tip)

    if not segments:
        return 0.0, 0.0

    # Sub-discretize all segments into sample points — vectorized
    all_xs = []
    all_ys = []
    for x0, y0, x1, y1 in segments:
        ts = np.linspace(0, 1, n_subsample, endpoint=False)
        all_xs.append(x0 + ts * (x1 - x0))
        all_ys.append(y0 + ts * (y1 - y0))

    # Include the final endpoint of the last segment
    all_xs.append(np.array([segments[-1][2]]))
    all_ys.append(np.array([segments[-1][3]]))

    obs_x = np.concatenate(all_xs)
    obs_y = np.concatenate(all_ys)

    # Vectorized orthogonal distance to parabola
    # Find closest point on parabola G*x^2 + b*x + c to each (obs_x, obs_y)
    xs = np.linspace(x_start, x_end, 1000)
    ys = gravity * xs**2 + b * xs + c

    # For each observed point, find minimum distance to any point on the curve
    # Shape: (n_observed, 1) - (1, n_curve) = (n_observed, n_curve)
    dx = obs_x[:, np.newaxis] - xs[np.newaxis, :]
    dy = obs_y[:, np.newaxis] - ys[np.newaxis, :]
    dists = np.sqrt(dx**2 + dy**2).min(axis=1)

    total_orthogonal = dists.sum()
    total_sq_orthogonal = (dists**2).sum()

    return total_orthogonal, total_sq_orthogonal



def main_root_length(arbor):
    """Total length of the main root, using edge 'length' attributes."""
    return sum(
        arbor[u][v]['length']
        for u, v in get_main_root_segments(arbor)
    )



def evaluate_parameters(arbor, G, alpha, cost_spec=HOMOGENEOUS):
    """
    Evaluate a single (G, alpha) combination for an already-loaded arbor graph.

    Returns
    -------
    tuple : (wiring, delay, total_orthogonal, total_sq_orthogonal)
    """
    results = arbor_best_cost(arbor, G, alpha, cost_spec=cost_spec)

    wiring = 0
    delay = 0
    total_orthogonal = 0
    total_sq_orthogonal = 0

    for result in results:
        wiring += result[1]
        delay += result[2]

        main_root_pt = (result[4], result[5])
        lateral_tip = (result[6], result[7])

        orth, sq_orth = calculate_orthogonal_errors(G, arbor, main_root_pt, lateral_tip)
        total_orthogonal += orth
        total_sq_orthogonal += sq_orth

    wiring += main_root_length(arbor)

    return wiring, delay, total_orthogonal, total_sq_orthogonal


# get_main_root_segments

def get_main_root_segments(arbor):
    """
    BFS from main root base along 'main root' labeled nodes, returning
    an ordered list of segment tuples ((x0, y0), (x1, y1)).
    """
    base = arbor.graph['main root base']

    segments = []
    visited = {base}
    queue = [base]

    while queue:
        curr = queue.pop(0)
        for neighbor in arbor.neighbors(curr):
            if neighbor in visited:
                continue
            if arbor.nodes[neighbor]['label'] in ('main root', 'main root base'):
                segments.append((curr, neighbor))
                visited.add(neighbor)
                queue.append(neighbor)

    return segments








# from pareto_functions.py 


def get_line_segment_drawings(line_segments, color="gray"):
    """Convert line segments into Plotly Scatter objects."""
    traces = []
    for seg in line_segments.values():
        (x0, y0), (x1, y1) = seg  # node IDs are tuples
        traces.append(go.Scatter(
            x=[x0, x1], y=[y0, y1],
            mode="lines",
            line=dict(color=color, width=4),
            showlegend=False
        ))
    return traces

def get_observed_lateral_segments(arbor):
    """
    Trace all lateral roots from tip to main root.
    Nodes are tuples (x, y).
    """
    segments = []
    tips = [n for n in arbor.nodes if arbor.nodes[n]["label"] == "lateral root tip"]

    for tip in tips:
        curr = tip
        prev = None
        while True:
            neighbors = list(arbor.neighbors(curr))
            if prev is not None:
                neighbors = [n for n in neighbors if n != prev]

            if len(neighbors) == 0:
                break  # end of lateral root

            next_node = neighbors[0]
            segments.append((curr, next_node))
            prev, curr = curr, next_node

            if arbor.nodes[curr]["label"].startswith("main root"):
                break

    return segments

def get_lateral_insertion_points(lateral_segments, arbor):
    """
    Find the coordinates where each lateral root meets the main root.
 
    These are the endpoints of lateral segments whose terminal node carries
    a 'main root' or 'main root base' label — i.e. the last node reached
    when tracing a lateral from its tip toward the main stem.
 
    Returns a list of (x, y) coordinate tuples, one per lateral root that
    successfully reaches the main root. Duplicate insertion points (two
    laterals sharing one junction) are included only once.

    (generated by Claude)
    """
    insertion_points = set()
    for seg in lateral_segments:
        node_a, node_b = seg
        if arbor.nodes[node_b]["label"].startswith("main root"):
            insertion_points.add(node_b)
    return list(insertion_points)

def get_lateral_segment_drawings(lateral_segments, color="lightgray"):
    traces = []
    for seg in lateral_segments:
        (x0, y0), (x1, y1) = seg
        traces.append(go.Scatter(
            x=[x0, x1], y=[y0, y1],
            mode="lines",
            line=dict(color=color, width=2),
            showlegend=False
        ))
    return traces

def get_tip_drawings(lat_tips, color="orange"):
    return [go.Scatter(x=[tip[0]], y=[tip[1]],
                       mode="markers",
                       marker=dict(color=color),
                       showlegend=False)
            for tip in lat_tips]

def get_insertion_point_drawings(insertion_points, color="blue"):
    """
    Draw lateral root insertion points (where a lateral meets the main root)
    as blue markers the same size as the lateral root tip markers.

    (generated by Claude)
    """
    return [go.Scatter(
                x=[pt[0]], y=[pt[1]],
                mode="markers",
                marker=dict(color=color, size=8),
                showlegend=False)
            for pt in insertion_points]

def get_lateral_nodes(lateral_segments):
    """
    Collect every unique node that appears anywhere along the lateral root
    segments — both endpoints of every segment.
 
    This covers all intermediate nodes along each lateral (the nodes between
    the tip and the insertion point), as well as the tips and insertion points
    themselves. Duplicates are removed so overlapping segment endpoints are
    not drawn twice.

    (generated by Claude)
    """
    nodes = set()
    for node_a, node_b in lateral_segments:
        nodes.add(node_a)
        nodes.add(node_b)
    return list(nodes)

def get_lateral_node_drawings(lateral_nodes, color="white"):
    """
    Draw every node along the lateral root segments as a default-sized marker.
    No explicit size is passed so Plotly uses its own default (6px).

    (generated by Claude)
    """
    return [go.Scatter(
                x=[node[0]], y=[node[1]],
                mode="markers",
                marker=dict(color=color),
                showlegend=False)
            for node in lateral_nodes]


def create_graphs(arbor, G, alpha):
    """
        Changed slightly for toy network
    """
    return arbor

def plot_arbors(arbor, G, alpha, show_observed=True, paper=False, save_fname=None):
    arbor_name = arbor.graph.get('arbor name', 'toy arbor')

    wiring, delay, total_orthogonal, total_sq_orthogonal = evaluate_parameters(arbor, G, alpha)

    print(f"\n→ G = {G}, alpha = {alpha}")
    print(f"Wiring cost: {wiring:.4f}")
    print(f"Conduction delay: {delay:.4f}\n")
    print(f"Total orthogonal distance: {total_orthogonal:.4f}")
    print(f"Total squared orthogonal distance: {total_sq_orthogonal:.4f}\n")

    fig = go.Figure()

    if show_observed:
        main_segments = get_main_root_segments(arbor)
        # convert list of tuples to dict for get_line_segment_drawings
        main_segments_dict = {i: seg for i, seg in enumerate(main_segments)}
        for trace in get_line_segment_drawings(main_segments_dict, color="black"):
            fig.add_trace(trace)

        lateral_segments = get_observed_lateral_segments(arbor)
        for trace in get_lateral_segment_drawings(lateral_segments, color="green"):
            fig.add_trace(trace)

        # --- all nodes along lateral root segments (white, default size) ---
        lateral_nodes = get_lateral_nodes(lateral_segments)
        for trace in get_lateral_node_drawings(lateral_nodes, color="white"):
            fig.add_trace(trace)

        lat_tips = [n for n in arbor.nodes if arbor.nodes[n]["label"] == "lateral root tip"]
        for trace in get_tip_drawings(lat_tips, color="orange"):
            fig.add_trace(trace)
        
        # --- lateral root insertion points (blue, same size as tips) ---
        insertion_points = get_lateral_insertion_points(lateral_segments, arbor)
        for trace in get_insertion_point_drawings(insertion_points, color="blue"):
            fig.add_trace(trace)

        base_node = arbor.graph['main root base']
        fig.add_trace(go.Scatter(
            x=[base_node[0]], y=[base_node[1]],
            mode="markers",
            marker=dict(color="purple", size=30),
            name="Main root base"
        ))

    fig.update_layout(
        title=f"Arbor: {arbor_name}   |   G={G}, alpha={alpha}",
        annotations=[
            dict(
                text="",
                xref="paper", yref="paper",
                x=0.5, y=-0.1,
                showarrow=False,
                font=dict(size=14)
            )
        ],
        xaxis_title="X",
        yaxis_title="Y",
        yaxis_autorange="reversed",
        width=850,
        height=700,
        margin=dict(t=80, b=80)
    )
    
    if paper:
        fig.update_layout(
            xaxis_title=None,
            yaxis_title=None,
            annotations=[], 
            title_text="", 
            showlegend=False
        )
        fig.update_xaxes(showticklabels=False)
        fig.update_yaxes(showticklabels=False)


    fig.show()
    
    if save_fname != None:
        print("saving fig to " + save_fname)
        fig.write_image(save_fname)



# Function that allows customizable test arbors 
def toy_arbor_gen(root, laterals, name='toy arbor'):
    root_nodes = list(root)
    if len(root_nodes) == 0:
        raise ValueError('root must contain at least one coordinate')

    G = nx.Graph()
    root_base = root_nodes[0]

    for r in root_nodes:
        G.add_node(r)
        G.nodes[r]['label'] = 'main root'

    G.nodes[root_base]['label'] = 'main root base'
    G.graph['main root base'] = root_base
    G.graph['arbor name'] = name

    for u, v in zip(root_nodes, root_nodes[1:]):
        connect_points(G, u, v)

    for lateral in laterals:
        G.add_node(lateral)
        G.nodes[lateral]['label'] = 'lateral root'
        connect_points(G, root_base, lateral)

    relabel_lateral_root_tips(G)

    return G

# Run above ^^

In [ ]:
toy = toy_network()
toy2 = toy_network2()
plot_arbors(toy2, 0, .1)

In [ ]:
# Test Cases

# 1 
root1 = [(0,0), (0,5)]
laterals1 = [(3,3)]
arbor1 = toy_arbor_gen(root1, laterals1, 'test1-single-lateral')
#plot_arbors(arbor1, 0, .1)

# 2
root2 = [(0,0), (0,10)]
laterals2 = [(2,2), (3,4), (1,6)]
arbor2 = toy_arbor_gen(root2, laterals2, 'test2-multi-lateral-same-base')
#plot_arbors(arbor2, 0, .1)

# 3
root3 = [(0,0), (5,0), (10,0), (15,0)]
laterals3 = [(2,4), (7,4), (12,4)]
arbor3 = toy_arbor_gen(root3, laterals3, 'test3-varying-to-root')
#plot_arbors(arbor3, 0, .1)

# 4
root4 = [(0,0), (0,5), (0,10)]
laterals4 = [(3,5), (-3,5)]  # equidistant on either side of midpoint
arbor4 = toy_arbor_gen(root4, laterals4, 'test4-symmetric')
#plot_arbors(arbor4, 0, .1)

# 5
root_coords = [(0, 0), (4, 1), (2, 0), (6, 3)]
lateral_coords = [(7, 5), (2, 3), (0, 5), (5, 0)]
toy_test = toy_arbor_gen(root_coords, lateral_coords, "toy-test")
#plot_arbors(toy_test, 0, .1)

test_arbors = [arbor1, arbor2, arbor3, arbor4, toy_test]

for arbor in test_arbors:
    name = arbor.graph['arbor name']
    # original = pf.conduction_delay(arbor)
    original = pg.gravitropism_conduction_delay(arbor)
    new_homogeneous = conduction_delay(arbor, cost_spec=pf.HOMOGENEOUS)

    print(f"\n{name}")
    print(f"  original:            {original:.6f}")
    print(f"  new (homogeneous):   {new_homogeneous:.6f}")
    print(f"  Do homogeneous match:   {abs(original - new_homogeneous) < 1e-9}")

In [ ]:

for arbor in test_arbors:
    name = arbor.graph['arbor name']
    # original = pf.conduction_delay(arbor)
    original = pg.gravitropism_conduction_delay(arbor)
    new_homogeneous = conduction_delay(arbor, cost_spec=pf.HOMOGENEOUS)
    heterogeneous_val = conduction_delay(arbor, HETEROGENEOUS)

    print(f"\n{name}")
    print(f"  original:            {original:.6f}")
    print(f"  new (homogeneous):   {new_homogeneous:.6f}")
    print(f"  Do homogeneous match:   {abs(original - new_homogeneous) < 1e-9}")
    print(f"  (Heterogeneous: {heterogeneous_val})")

In [ ]:
# Second round of testing for lateral_root_path_length in pareto_functions.py

lat_len_test_root = [(0,0), (4,5), (0, 10)]
lat_len_laterals = [(5, 5), (3,4), (1,6), (4, 2)]
lat_len_arbor = toy_arbor_gen(lat_len_test_root, lat_len_laterals, 'lateral-length-test-arbor')
plot_arbors(lat_len_arbor, 0, .1)

original = pg.gravitropism_conduction_delay(lat_len_arbor)
new_homogeneous = pf.conduction_delay(lat_len_arbor, cost_spec=pf.HOMOGENEOUS)
heterogeneous_val = conduction_delay(lat_len_arbor, HETEROGENEOUS)

print(f"\n{lat_len_arbor.graph['arbor name']}")
print(f"  original:            {original:.6f}")
print(f"  new (homogeneous):   {new_homogeneous:.6f}")
print(f"  Do homogeneous match:   {abs(original - new_homogeneous) < 1e-9}")
print(f"  (Heterogeneous: {heterogeneous_val})")



In [ ]:
# testing if all conduction delay versions match final value
pf_version = pf.conduction_delay(toy_test)
pf_initial_version = pf.conduction_delay_initial(toy_test)
localVer = conduction_delay(toy_test)


print(f"PF Version: {pf_version} and Local version: {localVer}")
print(pf_version == localVer)

print(f"PF INITIAL Version: {pf_initial_version} and Local version: {localVer}")
print(pf_initial_version == localVer)

In [ ]:
# Checking 
print(" ---- Conduction delay initial ----")
pf.conduction_delay_initial(toy_test)

print(" ---- New Conduction delay  ----")
pf.conduction_delay(toy_test)

In [2]:
# Adding duplicate coordinates to a networkx graph object

from pprint import pprint

fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt041_M058_10_S_1aba.csv"

points = [(5.617581, 3.265946), (5.589859, 3.536466), (5.637548, 3.365552), (5.610229, 3.47943), (5.6007, 3.5433), (5.573988, 3.605557), (5.637548, 3.365552), (5.489813, 4.056374), (5.585411, 3.738814), (5.530098, 3.944231), (5.432777, 4.306879), (5.616848, 5.052208), (5.577213, 4.966604), (5.501333, 4.761291), (5.691236, 5.323724), (5.771349, 5.73174), (5.795603, 6.123244), (5.801827, 6.364496), (5.835861, 6.431354), (5.633829, 3.303844)]

G = nx.Graph()
G.graph['arbor name'] = fname.strip('.csv')

i = 0
for point in points:
    if G.has_node(point):
        print(f"Duplicate point! >> {point}")

    G.add_node(i, coords=point, label='node label')

    print(f"Nodes in the graph: {G.nodes(data=True)}")
    i+=1

print("Adjacency dictionary")
pprint(G._adj)

print("Internal dictionary")
pprint(G.__dict__)

print("Node attributes dictionary")
pprint(G._node)

print(G.nodes[5]["coords"])



Nodes in the graph: [(0, {'coords': (5.617581, 3.265946), 'label': 'node label'})]
Nodes in the graph: [(0, {'coords': (5.617581, 3.265946), 'label': 'node label'}), (1, {'coords': (5.589859, 3.536466), 'label': 'node label'})]
Nodes in the graph: [(0, {'coords': (5.617581, 3.265946), 'label': 'node label'}), (1, {'coords': (5.589859, 3.536466), 'label': 'node label'}), (2, {'coords': (5.637548, 3.365552), 'label': 'node label'})]
Nodes in the graph: [(0, {'coords': (5.617581, 3.265946), 'label': 'node label'}), (1, {'coords': (5.589859, 3.536466), 'label': 'node label'}), (2, {'coords': (5.637548, 3.365552), 'label': 'node label'}), (3, {'coords': (5.610229, 3.47943), 'label': 'node label'})]
Nodes in the graph: [(0, {'coords': (5.617581, 3.265946), 'label': 'node label'}), (1, {'coords': (5.589859, 3.536466), 'label': 'node label'}), (2, {'coords': (5.637548, 3.365552), 'label': 'node label'}), (3, {'coords': (5.610229, 3.47943), 'label': 'node label'}), (4, {'coords': (5.6007, 3.543

In [ ]:
# testing read_arbor_full version that uses unique IDs for nodes
#fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt041_M058_10_S_1aba.csv"
fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt010_M248_1_C_10aba.csv"

G_initial = read_arbor_full_initial(fname)
G = read_arbor_full(fname)

main_root_base = G.graph.get('main root base', G.graph.get('main root'))
print(f"\n\n{G.nodes[9]['label']}")

print("Adjacency dictionary")
pprint(G._adj)

print("Internal dictionary")
pprint(G.__dict__)

print("Node attributes dictionary")
pprint(G._node)


vvvvvv Start of read_arbor_full vvvvvv
 [read_arbor_full] () This is curr_root if len(line) == 1: main root
 [read_arbor_full] () This is curr_root if len(line) == 1: 13bb6b78-171c-4e47-a41a-57be907a24e6
 [read_arbor_full] () This is curr_root if len(line) == 1: 50a20375-5aa8-4bc0-8b3d-424895f9dd65
 [read_arbor_full] () This is curr_root if len(line) == 1: d8a7da68-75bd-4bb7-ace8-70b49ed40f4b
 [read_arbor_full] () This is curr_root if len(line) == 1: 163ed70c-6b4c-4fe2-a16f-0b98ece09ea9
 [read_arbor_full] () This is curr_root if len(line) == 1: b0204bb8-d3ca-418f-bdee-5fd9e09b6505
 [read_arbor_full] () This is curr_root if len(line) == 1: 897a3936-ec8d-4c2f-8e4b-5e32fd4d0994
 [read_arbor_full] () This is curr_root if len(line) == 1: 21f45d3a-adba-4089-97f5-6c39824ad2f9
 [read_arbor_full] () This is curr_root if len(line) == 1: 4b49b981-daf2-4914-96e4-212b3138221c
 [read_arbor_full] () This is curr_root if len(line) == 1: ece193ed-2939-4746-87a8-6789c1ea76a2
 [read_arbor_full] () This i

In [4]:
# 6.30.26 and 7.1.26 testing new versions of conduction_delay and lateral_root_path_length


negatives = [
"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt009_M248_2_C_10aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt010_M248_1_C_10aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt041_M058_10_S_1aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt058_M248_3_S_1aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt059_M248_2_S_1aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt061_M248_7_C_1aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt063_M248_5_C_1aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt079_M058_6_S_noaba.csv" 
]



# No (-) to_root values: 

no_negatives = [
"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt014_M058_7_C_10aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt015_M058_6_C_10aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt039_M058_2_S_10aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt048_M058_3_S_1aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt049_M058_2_S_1aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt075_M058_10_S_noaba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt076_M058_9_S_noaba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt084_M058_1_S_noaba.csv", 

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt002_M248_9_C_10aba.csv",

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt003_M248_8_C_10aba.csv", 

"pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt004_M248_7_C_10aba.csv"
]



def testing_new_functions(G_initial, G, fname):
# wiring cost, making sure it still works
    homog_wiring = pf.wiring_cost(G_initial, cost_spec=HOMOGENEOUS)
    heterog_wiring = pf.wiring_cost(G_initial, cost_spec=HETEROGENEOUS)

    # conduction delay --> initial method, which only has a homogeneous calculation method
    initial_delay = pf.conduction_delay_initial(G_initial)

    print("(1/3) Completed Initial Wiring Cost and Conduction Delay ")

    # conduction delay --> uses new method of reading arbor file 
    homog_delay = pf.conduction_delay(G, cost_spec=HOMOGENEOUS)
    print(f"    (--/3) Homogeneous complete")
    heterog_delay = pf.conduction_delay(G, cost_spec=HETEROGENEOUS)
    print(f"    (--/3) Heterogeneous complete")
    print("(2/3) Completed New Conduction Delay")

    # conduction delay --> method from v2, using initial way of reading arbor file
    v2_homog_delay = pf.conduction_delay_v2(G_initial, cost_spec=HOMOGENEOUS)
    print(f"    (--/3) Homogeneous complete")
    v2_heterog_delay = pf.conduction_delay_v2(G_initial, cost_spec=HETEROGENEOUS)
    print(f"    (--/3) Heterogeneous complete")
    print("(3/3) Completed V2 Conduction Delay ")



    print(f"\nInitial Cost and Delay --> Homogeneous Wiring Cost: {homog_wiring}| Heterogeneous Wiring Cost: {heterog_wiring}")
    print(f"(MATCH HOPEFULLY) Homogeneous Conduction Delay: {initial_delay}| NO Heterogeneous Conduction Delay ---")
    print(f"\nV2 Delay --> Homogeneous Conduction Delay: {v2_homog_delay} | Homogeneous Conduction Delay: {v2_heterog_delay}")
    print(f"\nNew Version --> (MATCH HOPEFULLY) Homogeneous Conduction Delay: {homog_delay} | Heterogeneous Conduction Delay: {heterog_delay}")

print(f"-------- Looping Through Negatives ---------------")

for fname in negatives:
    G_initial = read_arbor_full_initial(fname)
    G = read_arbor_full(fname)
    print(f"\nReading through ---> {fname}")
    testing_new_functions(G_initial, G, fname)
    print("FINISHED\n")




print(f"\n\n...\n--------- Looping Through Non-Negatives ---------")
for fname in no_negatives:
    G_initial = read_arbor_full_initial(fname)
    G = read_arbor_full(fname)
    print(f"\nReading through ---> {fname}")
    testing_new_functions(G_initial, G, fname)
    print("FINISHED\n")


-------- Looping Through Negatives ---------------
------- p0: 0 | p1: 1 next_id: 76 | proj_point: (np.float64(5.685469385179388), np.float64(3.3013523896624872)) | lateral_start: 9 | lateral start coords: (5.690955, 3.299872)--------
------- p0: 0 | p1: 1 next_id: 77 | proj_point: (np.float64(5.761722655409148), np.float64(3.583910475477359)) | lateral_start: 65 | lateral start coords: (5.801581, 3.573154)--------
------- p0: 0 | p1: 1 next_id: 78 | proj_point: (np.float64(5.799071059889233), np.float64(3.7223057685098366)) | lateral_start: 73 | lateral start coords: (5.757724, 3.733464)--------
------- p0: 0 | p1: 1 next_id: 79 | proj_point: (np.float64(5.8173311793146825), np.float64(3.7899690225061535)) | lateral_start: 67 | lateral start coords: (5.810376, 3.791846)--------
------- p0: 0 | p1: 1 next_id: 80 | proj_point: (np.float64(5.819046058161191), np.float64(3.796323541896781)) | lateral_start: 71 | lateral start coords: (5.810681, 3.798581)--------
------- p0: 0 | p1: 1 next

In [ ]:
# 7.2.26 creating an attribute that pairs main root nodes with lateral tips

# fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt009_M248_2_C_10aba.csv"
# fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt011_M058_10_C_10aba.csv"
fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt041_M058_10_S_1aba.csv"
# fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt010_M248_1_C_10aba.csv"

G = read_arbor_full(fname)
nodes = list(G.nodes)

main_to_last = nx.shortest_path(G, source=0, target=nodes[-1])
print(main_to_last)

start = 20
end = 22
print(nx.shortest_path(G, source=start, target=end))

lat_tips = [
    n for n, data in G.nodes(data=True)
    if data.get("label") == "lateral root tip"
]
print(lat_tips)

print("Node attributes dictionary")
pprint(G._node)


# testing fifth version of lateral_root_path_length
tip = 20
#correct_length = G[82][34]['length'] + G[34][35]['length']
# correct_length = G[79][29]['length'] + G[29][30]['length'] + G[30][31]['length'] + G[31][32]['length'] + G[32][33]['length']
correct_length = G[20][19]['length'] + G[19][18]['length'] + G[18][17]['length'] + G[17][22]['length']

lrpl = pf.lateral_root_path_length(G, tip)
print(f"Lateral root path length: {lrpl}")
print(f"Should be: {correct_length}")

cond_delay = pf.conduction_delay()

------- p0: 1 | p1: 2 next_id: 21 | proj_point: (np.float64(6.043358209545036), np.float64(3.0061664011553835)) | lateral_start: 14 | lateral start coords: (6.039991, 3.008022)--------
------- p0: 1 | p1: 2 next_id: 22 | proj_point: (np.float64(6.043358209545036), np.float64(3.0061664011553835)) | lateral_start: 17 | lateral start coords: (6.039991, 3.008022)--------
------- p0: 1 | p1: 2 next_id: 23 | proj_point: (np.float64(6.18983494337172), np.float64(3.2719662305193338)) | lateral_start: 10 | lateral start coords: (6.217218, 3.256876)--------
------- p0: 1 | p1: 2 next_id: 24 | proj_point: (np.float64(6.1917043443814626), np.float64(3.275358485769852)) | lateral_start: 12 | lateral start coords: (6.219899, 3.259821)--------
[0, 1, 21, 22, 23, 24]
[20, 19, 18, 17, 22]
[11, 13, 16, 20]
Node attributes dictionary
{0: {'coords': (6.100607, 2.776608), 'label': 'main root base'},
 1: {'coords': (6.043083, 3.005667), 'label': 'main root'},
 2: {'coords': (6.611698, 4.037488), 'label': 'm

In [1]:
# fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt009_M248_2_C_10aba.csv"
# fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt011_M058_10_C_10aba.csv"
# fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt041_M058_10_S_1aba.csv"
fname = "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt010_M248_1_C_10aba.csv"

t_arbor = read_arbor_full(fname)
tree = nx.is_tree(t_arbor)
connected = nx.is_connected(t_arbor)

print(f"Arbor 10 Connected? {connected} | Arbor 10 Tree status: {tree}")
print("Done")

NameError: name 'read_arbor_full' is not defined

In [9]:
# Conduction delay calculated from a CSV file
import read_arbor_reconstruction as rar
fnames = ["pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt006_M248_5_C_10aba.csv", 
          
          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt009_M248_2_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt010_M248_1_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt041_M058_10_S_1aba.csv",

            "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt058_M248_3_S_1aba.csv",

            "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt059_M248_2_S_1aba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt061_M248_7_C_1aba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt063_M248_5_C_1aba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt079_M058_6_S_noaba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt080_M058_5_S_noaba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt094_M248_1_S_noaba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt095_M248_10_C_noaba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt097_M248_8_C_noaba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt098_M248_7_C_noaba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt104_M248_1_C_noaba.csv",

           "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt106_M058_9_C_noaba.csv",

           "pimpi_ABA_D9_set1_day9_20220611_RSA_M248M058LA1511_ABA_Salt127_LA1511_8_C_10aba.csv",

           "pimpi_ABA_D9_set1_day9_20220611_RSA_M248M058LA1511_ABA_Salt129_LA1511_6_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt007_M248_4_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt002_M248_9_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt008_M248_3_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt005_M248_6_C_10aba.csv", 

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt011_M058_10_C_10aba.csv"
          ]


prints = ["NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS",
          "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS",
          "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS",
          "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS",
          "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS",
          "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS", "NEGATIVE VALUE WARNINGS", 
          "No (-) warnings", "No (-) warnings", "No (-) warnings", "No (-) warnings", "No (-) warnings"]

count = 0

# ---- Checking if there are visual differences between arbors with (-) to_root values
for fname in fnames:
    arbor = read_arbor_full(fname)
    print(prints[count])
    plot_arbors(arbor, 0, .1)
    count += 1





 --- Appended (6.124498, 3.596031) to lateral_starts. Current status of lateral_starts: [(6.124498, 3.596031)] ---
 --- Appended (6.116114, 3.595936) to lateral_starts. Current status of lateral_starts: [(6.124498, 3.596031), (6.116114, 3.595936)] ---
 --- Appended (5.649386, 4.356012) to lateral_starts. Current status of lateral_starts: [(6.124498, 3.596031), (6.116114, 3.595936), (5.649386, 4.356012)] ---
 --- Appended (5.481423, 4.697907) to lateral_starts. Current status of lateral_starts: [(6.124498, 3.596031), (6.116114, 3.595936), (5.649386, 4.356012), (5.481423, 4.697907)] ---
 --- Appended (5.450674, 4.938667) to lateral_starts. Current status of lateral_starts: [(6.124498, 3.596031), (6.116114, 3.595936), (5.649386, 4.356012), (5.481423, 4.697907), (5.450674, 4.938667)] ---
 --- Appended (5.856709, 4.135888) to lateral_starts. Current status of lateral_starts: [(6.124498, 3.596031), (6.116114, 3.595936), (5.649386, 4.356012), (5.481423, 4.697907), (5.450674, 4.938667), (5.856

 --- Appended (5.690955, 3.299872) to lateral_starts. Current status of lateral_starts: [(5.690955, 3.299872)] ---
 --- Appended (5.811831, 3.800785) to lateral_starts. Current status of lateral_starts: [(5.690955, 3.299872), (5.811831, 3.800785)] ---
 --- Appended (5.818759, 3.82312) to lateral_starts. Current status of lateral_starts: [(5.690955, 3.299872), (5.811831, 3.800785), (5.818759, 3.82312)] ---
 --- Appended (5.95848, 4.346737) to lateral_starts. Current status of lateral_starts: [(5.690955, 3.299872), (5.811831, 3.800785), (5.818759, 3.82312), (5.95848, 4.346737)] ---
 --- Appended (6.030393, 4.618126) to lateral_starts. Current status of lateral_starts: [(5.690955, 3.299872), (5.811831, 3.800785), (5.818759, 3.82312), (5.95848, 4.346737), (6.030393, 4.618126)] ---
 --- Appended (5.954193, 4.33026) to lateral_starts. Current status of lateral_starts: [(5.690955, 3.299872), (5.811831, 3.800785), (5.818759, 3.82312), (5.95848, 4.346737), (6.030393, 4.618126), (5.954193, 4.330

 --- Appended (5.589859, 3.536466) to lateral_starts. Current status of lateral_starts: [(5.589859, 3.536466)] ---
 --- Appended (5.637548, 3.365552) to lateral_starts. Current status of lateral_starts: [(5.589859, 3.536466), (5.637548, 3.365552)] ---
 --- Appended (5.610229, 3.47943) to lateral_starts. Current status of lateral_starts: [(5.589859, 3.536466), (5.637548, 3.365552), (5.610229, 3.47943)] ---
 --- Appended (5.6007, 3.5433) to lateral_starts. Current status of lateral_starts: [(5.589859, 3.536466), (5.637548, 3.365552), (5.610229, 3.47943), (5.6007, 3.5433)] ---
 --- Appended (5.573988, 3.605557) to lateral_starts. Current status of lateral_starts: [(5.589859, 3.536466), (5.637548, 3.365552), (5.610229, 3.47943), (5.6007, 3.5433), (5.573988, 3.605557)] ---
 --- Appended (5.489813, 4.056374) to lateral_starts. Current status of lateral_starts: [(5.589859, 3.536466), (5.637548, 3.365552), (5.610229, 3.47943), (5.6007, 3.5433), (5.573988, 3.605557), (5.489813, 4.056374)] ---
 

 --- Appended (6.217218, 3.256876) to lateral_starts. Current status of lateral_starts: [(6.217218, 3.256876)] ---
 --- Appended (6.219899, 3.259821) to lateral_starts. Current status of lateral_starts: [(6.217218, 3.256876), (6.219899, 3.259821)] ---
 --- Appended (6.039991, 3.008022) to lateral_starts. Current status of lateral_starts: [(6.217218, 3.256876), (6.219899, 3.259821), (6.039991, 3.008022)] ---
vvvv Currently in connect_lateral_roots, checking if G is connected... vvv
^^^ End of connect_lateral_roots ^^^
In relabel_lateral_root_tips, is the graph still connected?:
Finished relabel_lateral_root_tips
NEGATIVE VALUE WARNINGS

→ G = 0, alpha = 0.1
Wiring cost: 9.8196
Conduction delay: 3.8647

Total orthogonal distance: 220.0950
Total squared orthogonal distance: 63.1945



 --- Appended (4.960601, 5.183151) to lateral_starts. Current status of lateral_starts: [(4.960601, 5.183151)] ---
 --- Appended (5.397835, 4.61432) to lateral_starts. Current status of lateral_starts: [(4.960601, 5.183151), (5.397835, 4.61432)] ---
 --- Appended (5.655723, 4.255028) to lateral_starts. Current status of lateral_starts: [(4.960601, 5.183151), (5.397835, 4.61432), (5.655723, 4.255028)] ---
 --- Appended (5.662709, 4.241774) to lateral_starts. Current status of lateral_starts: [(4.960601, 5.183151), (5.397835, 4.61432), (5.655723, 4.255028), (5.662709, 4.241774)] ---
 --- Appended (5.505459, 4.474193) to lateral_starts. Current status of lateral_starts: [(4.960601, 5.183151), (5.397835, 4.61432), (5.655723, 4.255028), (5.662709, 4.241774), (5.505459, 4.474193)] ---
 --- Appended (5.740769, 3.845061) to lateral_starts. Current status of lateral_starts: [(4.960601, 5.183151), (5.397835, 4.61432), (5.655723, 4.255028), (5.662709, 4.241774), (5.505459, 4.474193), (5.740769, 3

...

 --- Appended (5.228159, 5.245452) to lateral_starts. Current status of lateral_starts: [(5.228159, 5.245452)] ---
 --- Appended (5.693981, 3.620632) to lateral_starts. Current status of lateral_starts: [(5.228159, 5.245452), (5.693981, 3.620632)] ---
 --- Appended (5.745893, 3.612693) to lateral_starts. Current status of lateral_starts: [(5.228159, 5.245452), (5.693981, 3.620632), (5.745893, 3.612693)] ---
 --- Appended (5.737672, 3.081178) to lateral_starts. Current status of lateral_starts: [(5.228159, 5.245452), (5.693981, 3.620632), (5.745893, 3.612693), (5.737672, 3.081178)] ---
 --- Appended (5.497404, 3.304967) to lateral_starts. Current status of lateral_starts: [(5.228159, 5.245452), (5.693981, 3.620632), (5.745893, 3.612693), (5.737672, 3.081178), (5.497404, 3.304967)] ---
vvvv Currently in connect_lateral_roots, checking if G is connected... vvv
^^^ End of connect_lateral_roots ^^^
In relabel_lateral_root_tips, is the graph still connected?:
Finished relabel_lateral_root_ti

 --- Appended (5.723519, 3.178071) to lateral_starts. Current status of lateral_starts: [(5.723519, 3.178071)] ---
vvvv Currently in connect_lateral_roots, checking if G is connected... vvv
^^^ End of connect_lateral_roots ^^^
In relabel_lateral_root_tips, is the graph still connected?:
Finished relabel_lateral_root_tips
NEGATIVE VALUE WARNINGS

→ G = 0, alpha = 0.1
Wiring cost: 8.0124
Conduction delay: 3.0734

Total orthogonal distance: 578.3705
Total squared orthogonal distance: 738.4851



 --- Appended (6.185199, 3.826944) to lateral_starts. Current status of lateral_starts: [(6.185199, 3.826944)] ---
 --- Appended (6.165764, 4.067654) to lateral_starts. Current status of lateral_starts: [(6.185199, 3.826944), (6.165764, 4.067654)] ---
 --- Appended (6.1595, 4.271433) to lateral_starts. Current status of lateral_starts: [(6.185199, 3.826944), (6.165764, 4.067654), (6.1595, 4.271433)] ---
 --- Appended (6.126587, 4.584659) to lateral_starts. Current status of lateral_starts: [(6.185199, 3.826944), (6.165764, 4.067654), (6.1595, 4.271433), (6.126587, 4.584659)] ---
 --- Appended (5.986011, 5.489827) to lateral_starts. Current status of lateral_starts: [(6.185199, 3.826944), (6.165764, 4.067654), (6.1595, 4.271433), (6.126587, 4.584659), (5.986011, 5.489827)] ---
 --- Appended (5.931055, 5.641421) to lateral_starts. Current status of lateral_starts: [(6.185199, 3.826944), (6.165764, 4.067654), (6.1595, 4.271433), (6.126587, 4.584659), (5.986011, 5.489827), (5.931055, 5.641

 --- Appended (5.04221, 6.215542) to lateral_starts. Current status of lateral_starts: [(5.04221, 6.215542)] ---
 --- Appended (5.025002, 6.421193) to lateral_starts. Current status of lateral_starts: [(5.04221, 6.215542), (5.025002, 6.421193)] ---
 --- Appended (5.027041, 6.54855) to lateral_starts. Current status of lateral_starts: [(5.04221, 6.215542), (5.025002, 6.421193), (5.027041, 6.54855)] ---
 --- Appended (5.220762, 3.945097) to lateral_starts. Current status of lateral_starts: [(5.04221, 6.215542), (5.025002, 6.421193), (5.027041, 6.54855), (5.220762, 3.945097)] ---
 --- Appended (5.230347, 3.711737) to lateral_starts. Current status of lateral_starts: [(5.04221, 6.215542), (5.025002, 6.421193), (5.027041, 6.54855), (5.220762, 3.945097), (5.230347, 3.711737)] ---
 --- Appended (5.237973, 3.594047) to lateral_starts. Current status of lateral_starts: [(5.04221, 6.215542), (5.025002, 6.421193), (5.027041, 6.54855), (5.220762, 3.945097), (5.230347, 3.711737), (5.237973, 3.59404

KeyboardInterrupt: 

In [3]:
# Checking if there are value differences between versions of conduction_delay for certain CSVs

fnames = ["pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt041_M058_10_S_1aba.csv",
          
          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt006_M248_5_C_10aba.csv", 
          
          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt009_M248_2_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt010_M248_1_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt007_M248_4_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt002_M248_9_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt008_M248_3_C_10aba.csv",

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt005_M248_6_C_10aba.csv", 

          "pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt011_M058_10_C_10aba.csv"

          ]

for fname in fnames:
    arbor = read_arbor_full(fname)
    print()
    print()
    print(f"------- CONDUCTION DELAY CALLS FOR {fname} -------")
    print("Conduction Delay using lateral_root_path_length:")
    pf_version = pf.conduction_delay(arbor)
    print("ORIGINAL Conduction delay:")
    pf_initial_version = pf.conduction_delay_initial(arbor)
    print("Jupyter Notebook Conduction Delay Version:")
    localVer = conduction_delay(arbor)

    print()
    print(f"-- INFO FOR {fname} --")
    print(f"PF Version: {pf_version} and Local version: {localVer}")
    print(pf_version == localVer)

    print(f"PF INITIAL Version: {pf_initial_version} and Local version: {localVer}")
    print(pf_initial_version == localVer)


vvvvvv Start of read_arbor_full vvvvvv
vvvv Currently in connect_lateral_roots, checking if G is connected... vvv
^^^ End of connect_lateral_roots ^^^
In relabel_lateral_root_tips, is the graph still connected?:
Finished relabel_lateral_root_tips
^^^^^^ End of read_arbor_full ^^^^^^


------- CONDUCTION DELAY CALLS FOR pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt041_M058_10_S_1aba.csv -------
Conduction Delay using lateral_root_path_length:

========== conduction_delay: starting BFS from root (6.100607, 2.776608) ==========
  main root nodes (13 total): [(6.100607, 2.776608), (6.043083, 3.005667), (6.611698, 4.037488), (6.791123, 4.684194), (6.749546, 5.145985), (6.533428, 5.619591), (6.169345, 6.19738), (5.511777, 7.067731), (5.287211, 7.497938), (5.042015, 8.324716), (np.float64(6.043358209545036), np.float64(3.0061664011553835)), (np.float64(6.18983494337172), np.float64(3.2719662305193338)), (np.float64(6.1917043443814626), np.float64(3.275358485769852))]
  cost_spec

In [ ]:

'''
def toy_arbor_gen(root, laterals, name='toy arbor'):
    root_nodes = list(root)
    if len(root_nodes) == 0:
        raise ValueError('root must contain at least one coordinate')

    G = nx.Graph()
    root_base = root_nodes[0]

    for r in root_nodes:
        G.add_node(r)
        G.nodes[r]['label'] = 'main root'

    G.nodes[root_base]['label'] = 'main root base'
    G.graph['main root base'] = root_base
    G.graph['arbor name'] = name

    for u, v in zip(root_nodes, root_nodes[1:]):
        connect_points(G, u, v)

    for lateral in laterals:
        G.add_node(lateral)
        G.nodes[lateral]['label'] = 'lateral root'
        connect_points(G, root_base, lateral)

    relabel_lateral_root_tips(G)

    return G

    
for arbor in test_arbors:
    name = arbor.graph['arbor name']
    # original = pf.conduction_delay(arbor)
    original = pg.gravitropism_conduction_delay(arbor)
    new_homogeneous = conduction_delay(arbor, cost_spec=pf.HOMOGENEOUS)

    print(f"\n{name}")
    print(f"  original:            {original:.6f}")
    print(f"  new (homogeneous):   {new_homogeneous:.6f}")
    print(f"  Do homogeneous match:   {abs(original - new_homogeneous) < 1e-9}")
'''

def toy_arbor_gen_v2(root, laterals, name='toy arbor'):
    """
    Build a toy arbor graph where each lateral root can connect to any node
    on the main root, not only the root base.
 
    Parameters
    ----------
    root : list of (x, y) tuples
        Ordered nodes of the main root, root[0] is the base.
    laterals : list of (x, y) or ((x, y), (x, y)) tuples
        Each entry is either:
          - a bare (x, y) coord  → connected to root_base  (v1 behaviour)
          - ((tip_x, tip_y), (ins_x, ins_y)) → tip connected to ins point
          - ((tip_x, tip_y), None)            → connected to root_base
        The insertion point, when given, must appear in `root`.
    name : str
        Stored as G.graph['arbor name'].
 
    Returns
    -------
    G : nx.Graph with node labels and edge lengths set.
    """
    root_nodes = list(root)
    if len(root_nodes) == 0:
        raise ValueError('root must contain at least one coordinate')
 
    root_node_set = set(root_nodes)
 
    G = nx.Graph()
    root_base = root_nodes[0]
 
    # ---- build main root ----
    for r in root_nodes:
        G.add_node(r)
        G.nodes[r]['label'] = 'main root'
 
    G.nodes[root_base]['label'] = 'main root base'
    G.graph['main root base'] = root_base
    G.graph['arbor name'] = name
 
    for u, v in zip(root_nodes, root_nodes[1:]):
        connect_points(G, u, v)
 
    # ---- attach laterals ----
    for entry in laterals:
        # Detect format: bare coord vs (tip, insertion) pair
        if isinstance(entry[0], tuple):
            # entry is ((tip_x, tip_y), (ins_x, ins_y)) or ((tip_x, tip_y), None)
            tip, insertion = entry
        else:
            # entry is a plain (x, y) coord
            tip = entry
            insertion = None
 
        # Default insertion point is root_base (v1 behaviour)
        if insertion is None:
            insertion = root_base
 
        if insertion not in root_node_set:
            raise ValueError(
                f"Insertion point {insertion} is not in the main root. "
                f"Main root nodes: {root_nodes}"
            )
 
        G.add_node(tip)
        G.nodes[tip]['label'] = 'lateral root'
        connect_points(G, insertion, tip)
 
    relabel_lateral_root_tips(G)
    return G


def add_disconnected_pair(G, node_a, node_b):
    """
    Add two lateral root nodes connected only to each other (not to the main
    root). After relabelling, both ends become 'lateral root tip'.
 
    This reproduces the bug: lateral_root_path_length starts at one tip,
    finds the other tip labelled 'lateral root tip' (not 'main root'), and
    keeps walking — overcounting the path length — instead of stopping at
    an insertion point.
 
    get_insertion_segment will raise AssertionError for these nodes because
    no 'main root' node is ever reachable from an isolated pair. These
    arbors are intended to expose and test that failure path.
    """
    G.add_node(node_a)
    G.nodes[node_a]['label'] = 'lateral root'
    G.add_node(node_b)
    G.nodes[node_b]['label'] = 'lateral root'
    connect_points(G, node_a, node_b)
    relabel_lateral_root_tips(G)
    return G

# DISCONNECTED Toy Arbors (5)
disc_root1 = [(0, 0), (0, 3), (0, 6), (0, 9)]
disc_laterals1 = [
    ((2,  3), (0, 3)),   # inserts at the first non-base node
    ((-2, 6), (0, 6)),   # inserts at the mid node
    ((2,  9), (0, 9)),]
disc_arbor1 = toy_arbor_gen_v2(disc_root1, disc_laterals1, 'test6-one-disconnected-pair')
#add_disconnected_pair(disc_arbor1, (5, 1), (9, 1))

disc_root2 = [(0, 0), (2, 0), (4, 0), (6, 0), (8, 0), (10, 0)]
disc_laterals2 = [(2, 3), (5, 4), (8, 3), (10, 3)]
disc_arbor2 = toy_arbor_gen(disc_root2, disc_laterals2, 'test7-horizontal-one-disconnected-pair')

disc_root3 = [(0, 0), (0, 2), (0, 4), (0, 6), (0, 8), (0, 10), (0, 12), (0, 14)]
disc_laterals3 = [(3, 2), (-3, 6), (3, 10), (-3, 14)]
disc_arbor3 = toy_arbor_gen(disc_root3, disc_laterals3, 'test8-long-root-two-disconnected-pairs')

disc_root4 =  [(0,0), (3,1), (6,0), (9,1), (12,0), (15,1), (18,0), (21,1), (24,0), (27,1)]
disc_laterals4 = [(3, 5), (12, 5), (21, 5)]
disc_arbor4 = toy_arbor_gen(disc_root4, disc_laterals4, 'test9-zigzag-root-pair-near-base')

disc_root5 = [(0, 0), (0, 4), (0, 8), (0, 12), (0, 16)]
disc_laterals5 = [(3, 4), (-3, 8), (3, 12), (-3, 16)]
disc_arbor5 = toy_arbor_gen(disc_root5, disc_laterals5, 'test10-flanking-disconnected-pairs')

"""
# Toy Arbors with TIES (3)
tie_root1 = 
tie_laterals1 = 
tie_arbor1 = 


# SEMI-DISCONNECT toy arbor
semi_disc_root = 
semi_disc_laterals = 
semi_disc_arbor = 


# TIE + HALF-DISCONNECT toy arbor
tie_half_root = 
tie_half_laterals = 
tie_half_arbor = 
"""
all_toy_arbors = [disc_arbor1, disc_arbor2, disc_arbor3, disc_arbor4, disc_arbor5]

for arbor in all_toy_arbors:
    name = arbor.graph['arbor name']
    print(f"\n{name}")
    plot_arbors(arbor, 0, .1)



In [ ]:
# testing networkx functions

salt_041_arbor = read_arbor_full("pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt041_M058_10_S_1aba.csv")
salt_010_arbor = read_arbor_full("pimpi_ABA_D9_set1_day9_20220610_RSA_M248M058LA1511_ABA_Salt010_M248_1_C_10aba.csv")

base_041 = next(n for n, attr in salt_041_arbor.nodes(data=True) if attr.get('label') == 'main root base')
base_010 = next(n for n, attr in salt_010_arbor.nodes(data=True) if attr.get('label') == 'main root base')

main_root_041 = [n for n, attr in salt_041_arbor.nodes(data=True) if attr.get('label') == 'main root']
lat_tip_count_041 = sum(1 for n, d in salt_041_arbor.nodes(data=True) if d.get("label") == "lateral root tip")

main_root_010 = [n for n, attr in salt_010_arbor.nodes(data=True) if attr.get('label') == 'main root']
lat_tip_count_010 = sum(1 for n, d in salt_010_arbor.nodes(data=True) if d.get("label") == "lateral root tip")

# extracting final main root node
final_mainroot_041 = main_root_041[-1*lat_tip_count_041] # makes sure that list of main root nodes does not incorporate lateral root tip nodes that 
                                                         # still appear in the list even after applying the filter
final_mainroot_010 = main_root_010[-1*lat_tip_count_010]

path_041 = nx.shortest_path(salt_041_arbor, source=base_041, target=final_mainroot_041)
path_010 = nx.shortest_path(salt_010_arbor, source=base_010, target=final_mainroot_010)

print(f"Path for Salt 041 Arbor: {path_041}")
print(f"Path for Salt 010 Arbor: {path_010}")
